# Tiny LLM SFT Dataset Analysis on Google Colab

Runs an SFT dataset audit on a fresh Colab VM or locally in this repo.

**Flow:**
1. Detect Colab and optionally mount Drive.
2. Clone `40m-tiny-llm` on Colab.
3. Pick up an optional Hugging Face token.
4. Install `uv` and the notebook dependencies into the active Colab kernel.
5. Preprocess the full configured SFT dataset with the same pipeline as SFT training.
6. Analyze the actual preprocessed SFT cache: QA, instruction following, multiple choice, chat, summarization, translation, reasoning, cloze/next-word.
7. Compare the mix against the public Tiny LLM benchmarks: HellaSwag, WinoGrande, OpenBookQA, LAMBADA.
8. Export summary CSVs to Drive when running on Colab.

If you run this locally, the Colab-only cells skip themselves and the notebook uses the current checkout. The full preprocess step is enabled by default on Colab and disabled by default locally.

## 1. Runtime and Drive

On Colab this mounts Google Drive and creates a default output folder:

`drive/MyDrive/tiny_llm/sft_dataset_analysis`

Put raw datasets or copied SFT caches there if you want the notebook to find them from a fresh VM.

In [ ]:
from __future__ import annotations

import os
import pathlib

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/tiny_llm/sft_dataset_analysis")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Colab detected. Drive target base: {DRIVE_ROOT}")
else:
    DRIVE_ROOT = pathlib.Path.cwd() / "outputs" / "sft_dataset_analysis"
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Local runtime detected. Output base: {DRIVE_ROOT}")

## 2. Clone the repo on Colab

GitHub auth is read from `GH_TOKEN` / `GITHUB_TOKEN` or Colab Secrets with the same grant-access popup as `HF_TOKEN`. Add a Colab Secret named `GH_TOKEN` or `GITHUB_TOKEN` and grant notebook access when Colab asks.

In [ ]:
import os
import pathlib
import subprocess

REPO_OWNER = "marcbloech"
REPO_NAME = "40m-tiny-llm"
BRANCH = "post-training"

if IN_COLAB:
    REPO_DIR = pathlib.Path(f"/content/{REPO_NAME}")
    try:
        from google.colab import userdata
    except Exception:
        userdata = None

    def get_colab_secret(name):
        if userdata is None:
            return None
        try:
            return userdata.get(name)
        except Exception:
            return None

    GH_TOKEN = (
        os.environ.get("GH_TOKEN")
        or os.environ.get("GITHUB_TOKEN")
        or get_colab_secret("GH_TOKEN")
        or get_colab_secret("GITHUB_TOKEN")
    )
    if not GH_TOKEN:
        raise RuntimeError(
            "No GitHub token found. Add a Colab Secret named GH_TOKEN or GITHUB_TOKEN, "
            "then grant notebook access when Colab asks."
        )

    if REPO_DIR.exists():
        subprocess.run(["rm", "-rf", str(REPO_DIR)], check=True)

    public_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    clone_url = f"https://{GH_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git" if GH_TOKEN else public_url

    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", clone_url, str(REPO_DIR)],
        check=True,
    )
    if GH_TOKEN:
        subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", public_url], check=True)
        del GH_TOKEN
    del clone_url, public_url

    os.chdir(REPO_DIR)
    os.environ["PIKOGPT_REPO_DIR"] = str(REPO_DIR)
    print(f"Cloned into {REPO_DIR}")
    subprocess.run(["git", "log", "-1", "--oneline"], check=True)
else:
    REPO_DIR = pathlib.Path.cwd()
    os.environ["PIKOGPT_REPO_DIR"] = str(REPO_DIR)
    print(f"Using local checkout: {REPO_DIR}")

## 2b. Hugging Face token

The default SFT corpora are public, but setting `HF_TOKEN` avoids rate-limit or gated-dataset surprises if you add sources later.

Order of lookup:
1. Existing `HF_TOKEN` / `HUGGING_FACE_HUB_TOKEN` environment variable.
2. Colab secret named `HF_TOKEN`.
3. Interactive prompt; press Enter to skip.

In [ ]:
import getpass
import os

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

if IN_COLAB and not HF_TOKEN:
    try:
        from google.colab import userdata  # type: ignore
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if IN_COLAB and not HF_TOKEN:
    HF_TOKEN = getpass.getpass("HF token (Enter to skip): ").strip() or None

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print(f"HF token set in environment (len={len(HF_TOKEN)}).")
else:
    print("No HF token set. Public datasets should still work.")

## 3. Install dependencies into the active kernel

This differs slightly from the SFT training notebook: the analysis code runs inside the notebook kernel, so the project and notebook dependencies are installed into the active Colab Python environment.

In [ ]:
import os
import pathlib
import subprocess
import sys

if IN_COLAB:
    subprocess.run(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"], check=True)
    os.environ["PATH"] = f"{pathlib.Path.home() / '.local/bin'}:" + os.environ["PATH"]
    subprocess.run(["uv", "--version"], check=True)
    subprocess.run(
        ["uv", "pip", "install", "--system", "-e", ".[notebook]"],
        cwd=str(REPO_DIR),
        check=True,
    )
else:
    print("Skipping Colab dependency install. Use your local environment/kernel.")

print("Python:", sys.executable)

## Benchmark-Kontext

Das oeffentliche Tiny LLM-Leaderboard-Repo bewertet per CLI-Inference, nicht per interner Log-Likelihood. Die generierte Completion wird direkt geparst.

Public Benchmarks laut Repo:

| Benchmark | Public Split | Beispiele | Erwartetes Output-Format | Skill |
|---|---:|---:|---|---|
| HellaSwag | validation | 10'042 | `A`/`B`/`C`/`D` | plausibelste Fortsetzung / Commonsense |
| WinoGrande | validation | 1'267 | `A`/`B` | Coreference, Pronomen, Satzverstaendnis |
| OpenBookQA | validation | 500 | `A`/`B`/`C`/`D` | elementares Science-QA mit Multiple Choice |
| LAMBADA | test | 5'153 | erstes generiertes Wort | Long-context next-word prediction |

Leaderboard-Parsing:

- Multiple Choice: generiert standardmaessig bis zu 3 Tokens, nimmt den ersten nicht-whitespace Character und akzeptiert nur erlaubte Buchstaben.
- LAMBADA: generiert standardmaessig bis zu 5 Tokens, nimmt das erste Wort, normalisiert Klein-/Grossschreibung und Satzzeichen.
- `temperature=0`, `seed=0`, `--leaderboard`; stdout darf nur die Completion enthalten.

Baseline aus `leaderboard.csv` im Kurs-Repo: `public_avg=25.4`, HellaSwag `27.8`, LAMBADA `1.8`, OpenBookQA `22.6`, WinoGrande `49.4` bei `limit=500`.

Quelle: https://github.com/unisg-ics-dsnlp/Tiny LLM_Leaderboard

## Plan

1. Daten laden: lokaler SFT-Cache, eigene Raw-Datei oder optional die eingebauten Loader.
2. Standardspalten erzeugen: `instruction`, `input`, `output`, `source`, `split`, Laengen.
3. Task-Familien schaetzen: Multiple Choice, QA, General Instruction, Chat, Summarization, Translation, Reasoning, Code, Cloze.
4. Benchmark-Alignment messen: HellaSwag-like, WinoGrande-like, OpenBookQA-like, LAMBADA-like.
5. Qualitaetsrisiken markieren: Duplikate, leere Antworten, sehr lange Prompts, Truncation, Output-Format-Risiko.
6. Daraus einen Zielmix und konkrete SFT-Aenderungen ableiten.

In [ ]:
from __future__ import annotations

import json
import math
import random
import re
import sys
import warnings
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "src").exists():
            return p
    return start


env_repo = os.environ.get("PIKOGPT_REPO_DIR")
PROJECT_ROOT = Path(env_repo).resolve() if env_repo else find_project_root(Path.cwd())
SRC = PROJECT_ROOT / "src"
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 80)

print(f"project root: {PROJECT_ROOT}")

## 4. Dataset input

Choose one of these paths:

- `RAW_DATA_PATH`: a `.jsonl`, `.json`, `.csv`, or `.parquet` file.
- `SFT_CACHE_ROOT`: a directory containing Tiny LLM SFT cache files under `**/train.pt` and `**/val.pt`.
- `USE_BUILTIN_LOADERS=True`: load samples from the registered Tiny LLM loaders. The default list mirrors the current SFT mix: `alpaca`, `oasst1`, `dolly`, `arc`, `openbookqa`, `sciq`, `winogrande`, and `hellaswag`. This needs internet access and is best for quick inspection, not for auditing your final weighted dataset.

Fresh Colab defaults:

- raw file candidate: `/content/drive/MyDrive/tiny_llm/sft_dataset_analysis/my_sft.jsonl`
- SFT cache candidate: `/content/drive/MyDrive/tiny_llm/sft_dataset_analysis/sft`


In [ ]:
# Set this to a concrete raw file if you want to analyze your own already-prepared file.
# If RAW_DATA_PATH is None, the notebook builds the full SFT cache via configs/sft.toml
# and analyzes that cache, which mirrors the actual SFT training distribution
# (including dataset_weights oversampling and val_fraction split).
# Examples:
#   RAW_DATA_PATH = DRIVE_ROOT / "my_sft.jsonl"
#   RAW_DATA_PATH = PROJECT_ROOT / "data/my_sft.jsonl"
RAW_DATA_PATH = None

# Full preprocessing uses the same code path as SFT training:
# src/tiny_llm/posttraining/sft_dataset.py::prepare_sft_dataset
# Default is True so the analysis matches what training actually sees,
# both locally and on Colab.
PREPROCESS_FULL_DATASET = True
PREPROCESS_FORCE = False                # True ignores existing cache and rebuilds
PREPROCESS_CONFIG_PATH = PROJECT_ROOT / "configs" / "sft.toml"
PREPROCESS_MAX_SAMPLES = None           # None means full dataset, no per-source cap
PREPROCESS_CACHE_ON_DRIVE = IN_COLAB    # persistent across Colab sessions, slower than local disk

# Default cache roots. The Drive cache is useful on Colab because rebuilding full SFT data can take time.
LOCAL_SFT_CACHE_ROOT = PROJECT_ROOT / "data" / "sft"
DRIVE_SFT_CACHE_ROOT = DRIVE_ROOT / "sft"
PREPROCESS_CACHE_DIR = DRIVE_SFT_CACHE_ROOT if PREPROCESS_CACHE_ON_DRIVE else LOCAL_SFT_CACHE_ROOT
SFT_CACHE_ROOT = DRIVE_SFT_CACHE_ROOT if DRIVE_SFT_CACHE_ROOT.exists() else LOCAL_SFT_CACHE_ROOT

# Optional auto-detection for a single raw file in the Drive analysis folder.
if RAW_DATA_PATH is None and DRIVE_ROOT.exists():
    raw_candidates = []
    for pattern in ("*.jsonl", "*.json", "*.csv", "*.parquet", "*.pq"):
        raw_candidates.extend(sorted(DRIVE_ROOT.glob(pattern)))
    if len(raw_candidates) == 1:
        RAW_DATA_PATH = raw_candidates[0]
        PREPROCESS_FULL_DATASET = False
        print(f"Auto-selected raw dataset: {RAW_DATA_PATH}")
    elif len(raw_candidates) > 1:
        print("Multiple raw dataset candidates found. Set RAW_DATA_PATH explicitly:")
        for p in raw_candidates:
            print(" -", p)

# Builtin loaders pull raw, unweighted samples directly from HuggingFace and bypass
# the SFT preprocessing pipeline. Keep this off by default so the analysis reflects
# the real, weighted training distribution. Flip to True only for a quick sanity
# inspection of upstream loader behavior.
USE_BUILTIN_LOADERS = False
BUILTIN_DATASETS = ["alpaca", "oasst1", "dolly", "arc", "openbookqa", "sciq", "winogrande", "hellaswag"]
MAX_BUILTIN_SAMPLES_PER_DATASET = 2000  # HellaSwag also has its loader-internal cap for full builds

DECODE_TOKENIZED_CACHE = True
SAMPLE_FOR_DISPLAY = 3
MAX_TEXT_PREVIEW_CHARS = 320

print(f"RAW_DATA_PATH: {RAW_DATA_PATH}")
print(f"PREPROCESS_FULL_DATASET: {PREPROCESS_FULL_DATASET}")
print(f"USE_BUILTIN_LOADERS: {USE_BUILTIN_LOADERS}")
print(f"PREPROCESS_CACHE_DIR: {PREPROCESS_CACHE_DIR}")
print(f"SFT_CACHE_ROOT: {SFT_CACHE_ROOT}")

# ----------------------------------------------------------------------------
# Preprocessed-DataFrame snapshot
#
# After the slow path runs once (cache decode + tokenizer encode + classify),
# the fully labelled DataFrame is written to a parquet file under this folder.
# On subsequent runs the notebook loads that parquet directly and skips the
# expensive steps. Bump PREPROCESSED_SNAPSHOT_DATE whenever configs/sft.toml,
# the loader, or the classifier changes; otherwise the snapshot stays valid.
# Set PREPROCESS_FORCE=True (above) to ignore the snapshot and rebuild.
# ----------------------------------------------------------------------------
PREPROCESSED_SNAPSHOT_DATE = "29-04-2026-sft-mc-decontam"
if IN_COLAB:
    PREPROCESSED_SNAPSHOT_ROOT = pathlib.Path("/content/drive/MyDrive/tiny_llm/sft/preprocessed_data")
else:
    PREPROCESSED_SNAPSHOT_ROOT = PROJECT_ROOT / "outputs" / "sft_preprocessed_data"
PREPROCESSED_SNAPSHOT_DIR = PREPROCESSED_SNAPSHOT_ROOT / PREPROCESSED_SNAPSHOT_DATE
PREPROCESSED_DATAFRAME_PATH = PREPROCESSED_SNAPSHOT_DIR / "sft_dataframe.parquet"
print(f"PREPROCESSED_DATAFRAME_PATH: {PREPROCESSED_DATAFRAME_PATH}")


## 5. Preprocess the full SFT dataset

This cell builds the full preprocessed SFT dataset before analysis, using the same function as the training pipeline:

`prepare_sft_dataset(datasets, cache_dir, dataset_weights, max_seq_len, val_fraction, max_samples, seed, force)`

By default it reads `configs/sft.toml`, so it uses the configured datasets (`alpaca`, `oasst1`, `dolly`, `arc`, `openbookqa`, `sciq`, `winogrande`, `hellaswag`, `lambada_like`), weights, cleaning/filtering, public-eval decontamination, tokenization, train/val split, truncation handling and cache key. With `PREPROCESS_MAX_SAMPLES = None`, no global per-source debug cap is applied; HellaSwag still uses its loader-internal 6000-example cap and `lambada_like` uses its 5000-example cap.

The analysis that follows therefore sees the dataset as SFT training sees it: weighted train split plus unweighted validation split. This is intentional, because it answers "what distribution is the model actually trained on?" while preserving validation as a measurement set rather than a duplicated weighted sample.


In [ ]:
if PREPROCESS_FULL_DATASET and RAW_DATA_PATH is None:
    import tomllib
    from collections import Counter

    from tiny_llm.posttraining.sft_dataset import prepare_sft_dataset

    with open(PREPROCESS_CONFIG_PATH, "rb") as f:
        cfg = tomllib.load(f)
    post_cfg = cfg.get("posttraining", {})

    datasets = list(post_cfg.get("datasets", ["alpaca", "oasst1", "dolly", "arc", "openbookqa", "sciq", "winogrande", "hellaswag", "lambada_like"]))
    dataset_weights = dict(post_cfg.get("dataset_weights", {}))
    max_seq_len = int(post_cfg.get("max_seq_len", 1024))
    val_fraction = float(post_cfg.get("val_fraction", 0.02))

    PREPROCESS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print("Building/loading full preprocessed SFT dataset")
    print(f"- datasets: {datasets}")
    print(f"- weights: {dataset_weights}")
    print(f"- max_seq_len: {max_seq_len}")
    print(f"- val_fraction: {val_fraction}")
    print(f"- max_samples: {PREPROCESS_MAX_SAMPLES}")
    print(f"- cache_dir: {PREPROCESS_CACHE_DIR}")
    print(f"- force: {PREPROCESS_FORCE}")

    train_examples, val_examples = prepare_sft_dataset(
        datasets=datasets,
        cache_dir=PREPROCESS_CACHE_DIR,
        dataset_weights=dataset_weights,
        max_seq_len=max_seq_len,
        val_fraction=val_fraction,
        max_samples=PREPROCESS_MAX_SAMPLES,
        seed=SEED,
        force=PREPROCESS_FORCE,
    )
    SFT_CACHE_ROOT = PREPROCESS_CACHE_DIR

    def source_counts(examples):
        return Counter(getattr(ex, "source", "unknown") for ex in examples)

    def truncation_rate(examples):
        return sum(bool(getattr(ex, "truncated", False)) for ex in examples) / max(len(examples), 1)

    print(f"train examples: {len(train_examples):,}")
    print(f"val examples:   {len(val_examples):,}")
    train_source_counts = dict(source_counts(train_examples))
    val_source_counts = dict(source_counts(val_examples))
    mc_sources = {"arc", "openbookqa", "sciq", "winogrande", "hellaswag"}
    mc_train = sum(count for source, count in train_source_counts.items() if source in mc_sources)
    chat_train = train_source_counts.get("oasst1", 0)
    lambada_train = train_source_counts.get("lambada_like", 0)
    print("train by source:", train_source_counts)
    print("val by source:  ", val_source_counts)
    print(f"train MC/chat pct: {mc_train / max(len(train_examples), 1) * 100:.2f}% / {chat_train / max(len(train_examples), 1) * 100:.2f}%")
    print(f"train lambada_like pct: {lambada_train / max(len(train_examples), 1) * 100:.2f}%")
    print(f"train truncation rate: {truncation_rate(train_examples) * 100:.2f}%")
    print(f"val truncation rate:   {truncation_rate(val_examples) * 100:.2f}%")
elif RAW_DATA_PATH is not None:
    print("Skipping full preprocessing because RAW_DATA_PATH is set.")
else:
    print("Skipping full preprocessing. Set PREPROCESS_FULL_DATASET=True to build the complete SFT cache.")


## Loader

In [ ]:
def clean_cell(x) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and math.isnan(x):
        return ""
    return str(x).strip()


def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def extract_from_messages(messages) -> dict:
    """Best-effort mapping for ShareGPT/OpenAI-style chat records."""
    if not isinstance(messages, list):
        return {}

    user_turns = []
    assistant_turns = []
    for m in messages:
        if not isinstance(m, dict):
            continue
        role = clean_cell(m.get("role") or m.get("from")).lower()
        content = clean_cell(m.get("content") or m.get("value") or m.get("text"))
        if not content:
            continue
        if role in {"user", "human", "prompter"}:
            user_turns.append(content)
        elif role in {"assistant", "gpt", "bot"}:
            assistant_turns.append(content)

    if not user_turns or not assistant_turns:
        return {}

    history = []
    for u, a in zip(user_turns[:-1], assistant_turns[:-1]):
        history.append(f"User:\n{u}\n\nAssistant:\n{a}")
    return {
        "instruction": user_turns[-1],
        "input": "\n\n".join(history),
        "output": assistant_turns[-1],
    }


def normalise_record(row: dict, *, source: str, split: str, record_id: str | int | None = None) -> dict:
    if not isinstance(row, dict):
        row = {"text": str(row)}

    from_messages = extract_from_messages(row.get("messages") or row.get("conversations"))

    instruction = (
        row.get("instruction")
        or row.get("prompt")
        or row.get("question")
        or row.get("query")
        or from_messages.get("instruction")
        or ""
    )
    input_text = (
        row.get("input")
        or row.get("context")
        or row.get("passage")
        or row.get("document")
        or from_messages.get("input")
        or ""
    )
    output = (
        row.get("output")
        or row.get("response")
        or row.get("answer")
        or row.get("completion")
        or from_messages.get("output")
        or ""
    )

    return {
        "record_id": clean_cell(record_id if record_id is not None else row.get("id", "")),
        "source": clean_cell(row.get("source", source)) or source,
        "split": split,
        "instruction": clean_cell(instruction),
        "input": clean_cell(input_text),
        "output": clean_cell(output),
        "raw_keys": ",".join(sorted(map(str, row.keys()))),
        "truncated": bool(row.get("truncated", False)),
        "seq_len": row.get("seq_len"),
        "prompt_len": row.get("prompt_len"),
        "response_len": row.get("response_len"),
    }


def parse_alpaca_prompt(prompt_text: str) -> tuple[str, str]:
    instruction = ""
    input_text = ""
    m = re.search(
        r"### Instruction:\n(?P<instruction>.*?)(?:\n\n### Input:\n(?P<input>.*?))?\n\n### Response:\n?$",
        prompt_text,
        flags=re.DOTALL,
    )
    if m:
        instruction = clean_cell(m.group("instruction"))
        input_text = clean_cell(m.group("input") or "")
    return instruction, input_text


_TOKENIZER = None


def get_tokenizer():
    global _TOKENIZER
    if _TOKENIZER is not None:
        return _TOKENIZER
    try:
        from transformers import GPT2TokenizerFast

        _TOKENIZER = GPT2TokenizerFast.from_pretrained("gpt2")
        return _TOKENIZER
    except Exception as exc:
        warnings.warn(f"Could not load GPT-2 tokenizer. Text decode and token lengths may be missing: {exc}")
        return None


def load_raw_file(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    suffix = path.suffix.lower()

    if suffix == ".jsonl":
        rows = read_jsonl(path)
    elif suffix == ".json":
        obj = json.loads(path.read_text(encoding="utf-8"))
        rows = obj if isinstance(obj, list) else obj.get("data", obj.get("rows", []))
    elif suffix == ".csv":
        rows = pd.read_csv(path).to_dict("records")
    elif suffix in {".parquet", ".pq"}:
        rows = pd.read_parquet(path).to_dict("records")
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    records = [
        normalise_record(row, source=path.stem, split="raw", record_id=i)
        for i, row in enumerate(rows)
    ]
    return pd.DataFrame(records)


def load_sft_caches(cache_root: Path) -> pd.DataFrame:
    try:
        import torch
    except Exception as exc:
        warnings.warn(f"Cannot load torch SFT cache without torch: {exc}")
        return pd.DataFrame()

    pt_paths = [p for p in sorted(cache_root.glob("**/*.pt")) if p.name in {"train.pt", "val.pt"}]
    if not pt_paths:
        return pd.DataFrame()

    tok = get_tokenizer() if DECODE_TOKENIZED_CACHE else None
    records = []
    for pt_path in pt_paths:
        try:
            examples = torch.load(pt_path, map_location="cpu", weights_only=False)
        except Exception as exc:
            warnings.warn(f"Skipping {pt_path}: {exc}")
            continue

        split = pt_path.stem
        cache_name = pt_path.parent.name
        for i, ex in enumerate(examples):
            if not isinstance(ex, dict):
                ex = getattr(ex, "__dict__", {})
            ids = list(ex.get("input_ids") or [])
            prompt_len = int(ex.get("prompt_len") or 0)
            source = clean_cell(ex.get("source", "")) or cache_name

            instruction = ""
            input_text = ""
            output = ""
            if tok is not None and ids and prompt_len:
                prompt_text = tok.decode(ids[:prompt_len])
                response_text = tok.decode(ids[prompt_len:])
                response_text = response_text.split("<|endoftext|>", 1)[0]
                instruction, input_text = parse_alpaca_prompt(prompt_text)
                output = clean_cell(response_text)

            records.append(
                {
                    "record_id": f"{cache_name}:{split}:{i}",
                    "source": source,
                    "split": split,
                    "instruction": instruction,
                    "input": input_text,
                    "output": output,
                    "raw_keys": "input_ids,prompt_len,source,truncated",
                    "truncated": bool(ex.get("truncated", False)),
                    "seq_len": len(ids) if ids else None,
                    "prompt_len": prompt_len or None,
                    "response_len": max(len(ids) - prompt_len, 0) if ids and prompt_len else None,
                    "cache_path": (
                        str(pt_path.relative_to(PROJECT_ROOT))
                        if PROJECT_ROOT in pt_path.parents
                        else str(pt_path)
                    ),
                }
            )
    return pd.DataFrame(records)


def load_builtin_loader_samples() -> pd.DataFrame:
    from tiny_llm.posttraining.sft_dataset import DATASET_LOADERS

    rows = []
    for source in BUILTIN_DATASETS:
        loader = DATASET_LOADERS[source]
        for i, row in enumerate(loader(MAX_BUILTIN_SAMPLES_PER_DATASET, SEED)):
            rows.append(normalise_record(row, source=source, split="loader_sample", record_id=i))
    return pd.DataFrame(rows)

## Daten laden

In [ ]:
# Load the fully-preprocessed snapshot if one exists; otherwise fall through
# to the raw / builtin / SFT-cache loaders. `loaded_from_snapshot` is read by
# the metrics and classification cells below to skip work that is already
# baked into the snapshot.
loaded_from_snapshot = False
df = pd.DataFrame()

if PREPROCESSED_DATAFRAME_PATH.exists() and not PREPROCESS_FORCE:
    df = pd.read_parquet(PREPROCESSED_DATAFRAME_PATH)
    loaded_from_snapshot = True
    print(f"Loaded preprocessed snapshot ({len(df):,} rows): {PREPROCESSED_DATAFRAME_PATH}")
    print("Skipping re-tokenization and re-classification. "
          "Set PREPROCESS_FORCE=True or bump PREPROCESSED_SNAPSHOT_DATE to rebuild.")
elif RAW_DATA_PATH is not None:
    df = load_raw_file(Path(RAW_DATA_PATH))
elif USE_BUILTIN_LOADERS:
    df = load_builtin_loader_samples()
elif SFT_CACHE_ROOT.exists() and any(SFT_CACHE_ROOT.glob("**/*.pt")):
    df = load_sft_caches(SFT_CACHE_ROOT)
else:
    print("No data source found. Possible fixes:")
    print(f"  - Build the SFT cache: keep PREPROCESS_FULL_DATASET=True "
          f"(currently {PREPROCESS_FULL_DATASET}) and run the preprocess cell above.")
    print(f"  - Point RAW_DATA_PATH to a .jsonl/.json/.csv/.parquet file "
          f"(currently {RAW_DATA_PATH}).")
    print(f"  - Set USE_BUILTIN_LOADERS=True for a quick HuggingFace-only inspection.")
    print(f"  - Inspected SFT cache root: {SFT_CACHE_ROOT}")
    print(f"  - Looked for snapshot:    {PREPROCESSED_DATAFRAME_PATH}")

if df.empty:
    print("No rows loaded.")
else:
    print(f"loaded rows: {len(df):,}")
    display(df.head(3))


## Standard-Metriken und Task-Heuristiken

In [ ]:
def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", clean_cell(text)))


def short(text: str, n: int = MAX_TEXT_PREVIEW_CHARS) -> str:
    text = re.sub(r"\s+", " ", clean_cell(text))
    return text if len(text) <= n else text[: n - 1] + "..."


# Skip the heavy tokenizer-encode path when the snapshot already contains
# seq_len/prompt_len/response_len + word counts.
if not df.empty and not loaded_from_snapshot:
    for col in ["instruction", "input", "output", "source", "split"]:
        if col not in df:
            df[col] = ""
        df[col] = df[col].map(clean_cell)

    df["full_prompt"] = (df["instruction"] + "\n\n" + df["input"]).str.strip()
    df["full_text"] = (df["full_prompt"] + "\n\n" + df["output"]).str.strip()
    df["has_input"] = df["input"].str.len() > 0
    df["instruction_words"] = df["instruction"].map(word_count)
    df["input_words"] = df["input"].map(word_count)
    df["output_words"] = df["output"].map(word_count)
    df["total_words"] = df["full_text"].map(word_count)

    if "seq_len" not in df or df["seq_len"].isna().all():
        tok = get_tokenizer()
        if tok is not None:
            df["seq_len"] = df["full_text"].map(lambda x: len(tok.encode(x, add_special_tokens=False)))
            df["prompt_len"] = df["full_prompt"].map(lambda x: len(tok.encode(x, add_special_tokens=False)))
            df["response_len"] = df["output"].map(lambda x: len(tok.encode(x, add_special_tokens=False)))


In [ ]:
# ----------------------------------------------------------------------------
# Task & benchmark heuristics
#
# These regexes intentionally trade recall for precision: false-positive MC
# tags poison the bench-alignment view because the priority order in
# `classify_row` puts MC categories first. Each helper documents what it
# accepts and rejects.
# ----------------------------------------------------------------------------

# Multi-line option labels: a line that *starts* with "A)", "(B)", "1.", etc.,
# followed by real content. Inline letters inside prose ("plan A is...") and
# enumerated step lists buried in instructions are excluded by requiring the
# label to live at a line break.
OPTION_LINE_RE = re.compile(
    r"(?:^|\n)\s*[\(\[]?\s*([A-D]|[1-4])\s*[\)\].:\-]\s+\S",
)

# Single-letter MC answers like "A", "B.", "(C)", "D) Smith". We also accept
# a short trailing fragment (<=80 chars, no newline) so "B) Smith" still counts.
# Crucially we reject answers like "Apple is red" that merely start with A-D.
LETTER_ANSWER_RE = re.compile(
    r"^\s*[\(\[]?\s*([A-D])\s*(?:[\)\].:\-][^\n]{0,80})?\s*$",
    flags=re.IGNORECASE,
)

QUESTION_START = re.compile(
    r"^\s*(what|why|how|when|where|who|which|is|are|do|does|did|can|could|"
    r"would|should|was|were)\b",
    flags=re.IGNORECASE,
)
SCIENCE_TERMS = re.compile(
    r"\b(force|energy|matter|organism|planet|chemical|temperature|water|light|"
    r"sound|earth|animal|plant|cell|gas|solid|liquid|ecosystem|electric|magnet|"
    r"gravity|photosynthesis|biology|physics|chemistry|atom|molecule|species)\b",
    flags=re.IGNORECASE,
)
PRONOUN_RE = re.compile(
    r"\b(he|she|it|they|him|her|them|his|hers|its|their)\b",
    flags=re.IGNORECASE,
)
CODE_FENCE_RE = re.compile(r"```[\s\S]*?```|`[^`\n]+`")
# Code hints are matched as full tokens / phrases. Bare "function" is dropped
# because phrases like "describe the function of mitochondria" used to leak
# into the coding bucket.
CODE_HINTS_RE = re.compile(
    r"\b(?:python|javascript|typescript|c\+\+|c#|sql|html|css|regex|bash|"
    r"shell\s+script|kotlin|rust|golang|scala|"
    r"def\s+\w+\s*\(|function\s+\w+\s*\(|class\s+[A-Z]\w*\b|"
    r"import\s+[\w\.]+|console\.log|print\s*\(|return\s+[\w\(\[\"\']|"
    r"write\s+(?:a\s+|the\s+)?(?:python|javascript|typescript|java|c\+\+|"
    r"sql|html|css|bash|shell|regex|code|program|function|script|class|method)|"
    r"debug\s+(?:this|the)\s+code|fix\s+(?:this|the)\s+code|"
    r"implement\s+(?:a|the)\s+\w+\s+(?:function|class|method))\b",
    flags=re.IGNORECASE,
)


def has_options(text: str) -> bool:
    """True if the text presents at least two distinct labelled MC options."""
    text = clean_cell(text)
    if not text:
        return False
    if "multiple choice" in text.lower():
        return True
    matches = OPTION_LINE_RE.findall(text)
    if not matches:
        return False
    seen = {m.upper() for m in matches}
    letters = seen & {"A", "B", "C", "D"}
    numbers = seen & {"1", "2", "3", "4"}
    return len(letters) >= 2 or len(numbers) >= 2


def output_is_letter(text: str) -> bool:
    """True if the output is a single-letter MC answer with optional short suffix."""
    text = clean_cell(text)
    if not text or len(text) > 120:
        return False
    return bool(LETTER_ANSWER_RE.match(text))


CHAT_SOURCES = {"oasst1", "oasst", "sharegpt", "ultrachat"}
MC_SOURCES = {"arc", "openbookqa", "sciq", "winogrande", "hellaswag"}
LAMBADA_SOURCES = {"lambada_like"}


def classify_row(row) -> dict:
    instruction = clean_cell(row.get("instruction"))
    input_text = clean_cell(row.get("input"))
    output = clean_cell(row.get("output"))
    source = clean_cell(row.get("source")).lower()
    prompt = f"{instruction}\n\n{input_text}".strip()
    prompt_low = prompt.lower()
    input_low = input_text.lower()

    multiple_choice = source in MC_SOURCES or has_options(prompt) or output_is_letter(output)

    qa = ("?" in prompt) or bool(QUESTION_START.search(instruction))

    chat = (
        "previous conversation:" in input_low
        or ("user:" in input_low and "assistant:" in input_low)
        or source in CHAT_SOURCES
    )

    summarization = any(k in prompt_low for k in [
        "summarize", "summarise", "summary of", "tl;dr", "in one sentence",
        "in a sentence", "in a few sentences",
    ])
    translation = (
        bool(re.search(r"\btranslat\w+\b", prompt_low))
        or any(k in prompt_low for k in [
            "uebersetze", "übersetze", "traduce", "traduire", "in english:", "in german:",
        ])
    )
    reasoning_math = any(k in prompt_low for k in [
        "step by step", "let's think", "let us think", "calculate", "compute the",
        "solve for", "equation", "math problem", "logic puzzle", "prove that",
        "derivation", "what is the value of", "how many",
    ])
    coding = bool(CODE_FENCE_RE.search(prompt)) or bool(CODE_HINTS_RE.search(prompt))
    extraction = any(k in prompt_low for k in [
        "extract ", "find all ", "list all ", "identify ", "return only ",
        "extract the ", "as json", "in json format", "json output",
    ])
    classification = any(k in prompt_low for k in [
        "classify", "label this", "label the", "categorize", "categorise",
        "sentiment of", "true or false", "is this positive", "is this negative",
        "is the following",
    ])
    creative = any(k in prompt_low for k in [
        "write a story", "write a poem", "write a song", "creative writing",
        "compose a poem", "compose a song", "write a dialogue", "write a fictional",
        "write a short story",
    ])
    cloze = (
        source in LAMBADA_SOURCES
        or any(k in prompt_low for k in [
            "fill in the blank", "next word", "predict the single word", "one word",
            "complete the sentence", "complete the passage", "missing word",
        ])
        or "___" in prompt
        or " _ " in prompt
    )

    hellaswag_like = source == "hellaswag" or (multiple_choice and any(k in prompt_low for k in [
        "what happens next", "best ending", "most likely ending", "continuation",
        "complete the story", "what would happen", "most plausible",
    ]))
    winogrande_like = source == "winogrande" or (multiple_choice and (
        "refers to" in prompt_low
        or (("___" in prompt or "blank" in prompt_low) and bool(PRONOUN_RE.search(prompt)))
    ))
    openbookqa_like = source in {"arc", "openbookqa", "sciq"} or (
        multiple_choice
        and (qa or bool(SCIENCE_TERMS.search(prompt)))
        and not hellaswag_like
        and not winogrande_like
    )
    lambada_like = (
        source in LAMBADA_SOURCES
        or cloze
        or "next word" in prompt_low
        or (
            int(row.get("input_words") or 0) >= 60
            and int(row.get("output_words") or 0) <= 2
            and not multiple_choice
        )
    )

    specific = [
        multiple_choice, qa, chat, summarization, translation,
        reasoning_math, coding, extraction, classification, creative, cloze,
    ]
    general_instruction = bool(instruction) and not any(specific)

    # Priority order: benchmark-shaped MC first (most actionable for leaderboard),
    # then cloze (LAMBADA), then format/source-driven buckets (chat, code), then
    # task-keyword buckets, then plain QA, then generic instruction.
    if hellaswag_like:
        primary = "hellaswag_like_mc"
    elif winogrande_like:
        primary = "winogrande_like_mc"
    elif openbookqa_like:
        primary = "openbookqa_like_mc"
    elif multiple_choice:
        primary = "multiple_choice"
    elif lambada_like:
        primary = "cloze_next_word"
    elif chat:
        primary = "chat"
    elif coding:
        primary = "coding"
    elif reasoning_math:
        primary = "reasoning_math"
    elif summarization:
        primary = "summarization"
    elif translation:
        primary = "translation"
    elif extraction:
        primary = "extraction"
    elif classification:
        primary = "classification"
    elif creative:
        primary = "creative_writing"
    elif qa:
        primary = "open_qa"
    elif general_instruction:
        primary = "general_instruction"
    else:
        primary = "unknown"

    return {
        "primary_task": primary,
        "multiple_choice": multiple_choice,
        "qa": qa,
        "general_instruction": general_instruction,
        "chat": chat,
        "summarization": summarization,
        "translation": translation,
        "reasoning_math": reasoning_math,
        "coding": coding,
        "extraction": extraction,
        "classification": classification,
        "creative_writing": creative,
        "cloze_next_word": cloze,
        "bench_hellaswag_like": hellaswag_like,
        "bench_winogrande_like": winogrande_like,
        "bench_openbookqa_like": openbookqa_like,
        "bench_lambada_like": lambada_like,
    }


if not df.empty and not loaded_from_snapshot:
    labels = df.apply(classify_row, axis=1, result_type="expand")
    df = pd.concat([df, labels], axis=1)
    PREPROCESSED_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_parquet(PREPROCESSED_DATAFRAME_PATH, index=False)
    print(f"Saved preprocessed snapshot: {PREPROCESSED_DATAFRAME_PATH}")

if not df.empty:
    display(df[["source", "split", "primary_task", "instruction", "input", "output"]].head(5))


## Uebersicht: Quellen, Splits, Task-Mix

## Dataset size and token statistics

This is the core size check: number of samples, total tokens, average/median/p95 tokens per sample, prompt-vs-response tokens, split/source totals, and an optimizer-step estimate for SFT.

In [ ]:
if df.empty:
    print("Load data first.")
else:
    token_cols = ["seq_len", "prompt_len", "response_len"]
    for col in token_cols:
        if col not in df:
            df[col] = pd.NA
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if df["seq_len"].notna().any():
        size_summary = {
            "samples_total": int(len(df)),
            "samples_train": int((df["split"] == "train").sum()),
            "samples_val": int((df["split"] == "val").sum()),
            "tokens_total": int(df["seq_len"].sum()),
            "tokens_train": int(df.loc[df["split"] == "train", "seq_len"].sum()),
            "tokens_val": int(df.loc[df["split"] == "val", "seq_len"].sum()),
            "tokens_per_sample_mean": round(float(df["seq_len"].mean()), 2),
            "tokens_per_sample_median": round(float(df["seq_len"].median()), 2),
            "tokens_per_sample_p90": round(float(df["seq_len"].quantile(0.90)), 2),
            "tokens_per_sample_p95": round(float(df["seq_len"].quantile(0.95)), 2),
            "tokens_per_sample_p99": round(float(df["seq_len"].quantile(0.99)), 2),
            "tokens_per_sample_max": int(df["seq_len"].max()),
            "prompt_tokens_total": int(df["prompt_len"].fillna(0).sum()),
            "response_tokens_total": int(df["response_len"].fillna(0).sum()),
            "response_token_share_pct": round(
                float(df["response_len"].fillna(0).sum() / max(df["seq_len"].sum(), 1) * 100),
                2,
            ),
        }
        display(pd.Series(size_summary).to_frame("value"))

        print("Token distribution by split")
        split_stats = (
            df.groupby("split")
            .agg(
                samples=("seq_len", "size"),
                tokens_total=("seq_len", "sum"),
                tokens_mean=("seq_len", "mean"),
                tokens_median=("seq_len", "median"),
                tokens_p95=("seq_len", lambda s: s.quantile(0.95)),
                response_tokens_mean=("response_len", "mean"),
                response_token_share_pct=("response_len", lambda s: s.fillna(0).sum()),
            )
        )
        split_token_totals = df.groupby("split")["seq_len"].sum().replace(0, pd.NA)
        split_stats["response_token_share_pct"] = (
            split_stats["response_token_share_pct"] / split_token_totals * 100
        )
        display(split_stats.round(2))

        print("Token distribution by source")
        source_stats = (
            df.groupby("source")
            .agg(
                samples=("seq_len", "size"),
                tokens_total=("seq_len", "sum"),
                tokens_share_pct=("seq_len", lambda s: s.sum() / max(df["seq_len"].sum(), 1) * 100),
                tokens_mean=("seq_len", "mean"),
                tokens_median=("seq_len", "median"),
                tokens_p95=("seq_len", lambda s: s.quantile(0.95)),
                response_tokens_mean=("response_len", "mean"),
            )
            .sort_values("tokens_total", ascending=False)
        )
        display(source_stats.round(2))

        # Optimizer-step estimate for the configured SFT defaults.
        try:
            import tomllib

            with open(PREPROCESS_CONFIG_PATH, "rb") as f:
                cfg = tomllib.load(f)
            post_cfg = cfg.get("posttraining", {})
            batch_size = int(post_cfg.get("batch_size", 16))
            grad_accum = int(post_cfg.get("grad_accum_steps", 1))
            epochs = float(post_cfg.get("epochs", 1))
            train_samples = max(int((df["split"] == "train").sum()), 0)
            optim_steps = math.floor(train_samples / max(batch_size, 1) / max(grad_accum, 1) * epochs)
            tokens_per_step_mean = float(
                df.loc[df["split"] == "train", "seq_len"].mean() * batch_size * grad_accum
            )
            train_token_passes = float(
                df.loc[df["split"] == "train", "seq_len"].sum() * epochs
            )
            estimate = {
                "config_batch_size": batch_size,
                "config_grad_accum_steps": grad_accum,
                "config_epochs": epochs,
                "estimated_optimizer_steps": optim_steps,
                "mean_tokens_per_optimizer_step": round(tokens_per_step_mean, 2),
                "train_token_passes_over_epochs": int(train_token_passes),
            }
            print("Training-size estimate from configs/sft.toml")
            display(pd.Series(estimate).to_frame("value"))
        except Exception as exc:
            print(f"Could not compute config-based training estimate: {exc}")
    else:
        print("No token columns available. This should only happen if tokenizer loading failed for raw text.")

In [ ]:
def pct_table(counts: pd.Series, denom: int | None = None) -> pd.DataFrame:
    denom = int(denom if denom is not None else counts.sum())
    out = counts.rename("count").to_frame()
    out["pct"] = (out["count"] / max(denom, 1) * 100).round(2)
    return out


if df.empty:
    print("Load data first.")
else:
    unknown_count = int((df["primary_task"] == "unknown").sum())
    overview = {
        "rows": len(df),
        "sources": df["source"].nunique(),
        "splits": ", ".join(sorted(df["split"].dropna().unique())),
        "has_input_pct": round(df["has_input"].mean() * 100, 2),
        "empty_instruction": int((df["instruction"].str.len() == 0).sum()),
        "empty_output": int((df["output"].str.len() == 0).sum()),
        "duplicate_full_text_pct": round(df["full_text"].duplicated().mean() * 100, 2),
        "truncated_pct": round(df.get("truncated", pd.Series(False, index=df.index)).astype(bool).mean() * 100, 2),
        "unknown_task_pct": round(unknown_count / max(len(df), 1) * 100, 2),
    }
    display(pd.Series(overview).to_frame("value"))

    if overview["unknown_task_pct"] > 5:
        print(
            f"WARN: {unknown_count} samples ({overview['unknown_task_pct']}%) "
            "fell into 'unknown'. The heuristics may be missing a category."
        )

    print("source x split")
    display(pd.crosstab(df["source"], df["split"], margins=True))

    print("primary task distribution")
    task_mix = pct_table(df["primary_task"].value_counts(), len(df))
    display(task_mix)

    print("primary task by source (rows = source, columns = task, % within source)")
    source_task = (
        pd.crosstab(df["source"], df["primary_task"], normalize="index")
        .mul(100)
        .round(1)
        .sort_index()
    )
    display(source_task)


In [ ]:
if not df.empty:
    flag_cols = [
        "multiple_choice",
        "qa",
        "general_instruction",
        "chat",
        "summarization",
        "translation",
        "reasoning_math",
        "coding",
        "extraction",
        "classification",
        "creative_writing",
        "cloze_next_word",
    ]
    flag_mix = pd.DataFrame(
        {
            "count": df[flag_cols].sum().astype(int),
            "pct": (df[flag_cols].mean() * 100).round(2),
        }
    ).sort_values("pct", ascending=False)
    display(flag_mix)

## Benchmark-Alignment

In [ ]:
if not df.empty:
    bench_cols = [
        "bench_hellaswag_like",
        "bench_winogrande_like",
        "bench_openbookqa_like",
        "bench_lambada_like",
    ]
    bench_mix = pd.DataFrame(
        {
            "count": df[bench_cols].sum().astype(int),
            "pct": (df[bench_cols].mean() * 100).round(2),
        }
    )
    display(bench_mix)

    by_source = df.groupby("source")[bench_cols].mean().mul(100).round(2)
    display(by_source.sort_index())

    # MC format compliance: for each MC-shaped bucket, what fraction of outputs
    # is actually a leaderboard-parseable single-letter answer? Verbose answers
    # ("The correct answer is C, because...") don't pass the leaderboard's
    # first-character parser, so a low compliance rate here is a red flag.
    mc_cols = [
        "multiple_choice",
        "bench_hellaswag_like",
        "bench_winogrande_like",
        "bench_openbookqa_like",
    ]
    letter_only_re = r"^\s*[\(\[]?\s*[A-D](?:[\)\].:\-]\s*\S*)?\s*$"
    output_is_clean_letter = df["output"].str.match(letter_only_re, case=False, na=False)
    rows = []
    for col in mc_cols:
        subset = df[df[col]]
        n = int(len(subset))
        clean = int(output_is_clean_letter[df[col]].sum())
        rows.append(
            {
                "bucket": col,
                "samples": n,
                "letter_only_outputs": clean,
                "compliance_pct": round(clean / max(n, 1) * 100, 2),
            }
        )
    print("MC output format compliance (single-letter answer rate)")
    display(pd.DataFrame(rows).set_index("bucket"))


In [ ]:
if not df.empty:
    mc_sources = {"arc", "openbookqa", "sciq", "winogrande", "hellaswag"}
    mc_mask = df["source"].isin(mc_sources) | df["multiple_choice"].fillna(False)
    mc_df = df[mc_mask].copy()
    mc_df["letter_only_output"] = mc_df["output"].map(output_is_letter)
    compliance = pd.DataFrame(
        {
            "count": mc_df.groupby("source").size(),
            "letter_only_count": mc_df.groupby("source")["letter_only_output"].sum().astype(int),
        }
    ).fillna(0).astype(int)
    compliance["letter_only_pct"] = (
        compliance["letter_only_count"] / compliance["count"].clip(lower=1) * 100
    ).round(2)
    print(f"MC compliance overall: {int(mc_df['letter_only_output'].sum()):,} / {len(mc_df):,} ({mc_df['letter_only_output'].mean() * 100:.2f}%)")
    display(compliance.sort_index())


## Visualisierung

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

    df["primary_task"].value_counts().sort_values().plot(kind="barh", ax=axes[0], color="#2b6cb0")
    axes[0].set_title("Primary task count")
    axes[0].set_xlabel("examples")

    df["source"].value_counts().sort_values().plot(kind="barh", ax=axes[1], color="#2f855a")
    axes[1].set_title("Source count")
    axes[1].set_xlabel("examples")

    bench_pct = df[[
        "bench_hellaswag_like",
        "bench_winogrande_like",
        "bench_openbookqa_like",
        "bench_lambada_like",
    ]].mean().mul(100).sort_values()
    bench_pct.plot(kind="barh", ax=axes[2], color="#b7791f")
    axes[2].set_title("Benchmark-like share")
    axes[2].set_xlabel("percent")

    plt.tight_layout()

In [ ]:
if not df.empty:
    heat = pd.crosstab(df["source"], df["primary_task"], normalize="index").mul(100).round(1)
    display(heat)

    fig, ax = plt.subplots(figsize=(min(18, 2 + 0.65 * heat.shape[1]), max(4, 0.45 * heat.shape[0])))
    im = ax.imshow(heat.values, aspect="auto", cmap="Blues")
    ax.set_xticks(range(heat.shape[1]), heat.columns, rotation=45, ha="right")
    ax.set_yticks(range(heat.shape[0]), heat.index)
    ax.set_title("Task mix per source (%)")
    plt.colorbar(im, ax=ax, fraction=0.025)
    plt.tight_layout()

## Laengen, Truncation und Format-Risiken

In [ ]:
if not df.empty:
    length_cols = [c for c in ["seq_len", "prompt_len", "response_len", "instruction_words", "input_words", "output_words"] if c in df]
    display(df[length_cols].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(2))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    if "seq_len" in df and df["seq_len"].notna().any():
        df["seq_len"].dropna().clip(upper=df["seq_len"].quantile(0.99)).hist(bins=50, ax=axes[0], color="#4a5568")
        axes[0].set_title("Sequence length, clipped at p99")
        axes[0].set_xlabel("tokens")
    else:
        axes[0].axis("off")

    df["output_words"].clip(upper=df["output_words"].quantile(0.99)).hist(bins=50, ax=axes[1], color="#805ad5")
    axes[1].set_title("Output length, clipped at p99")
    axes[1].set_xlabel("words")
    plt.tight_layout()

    risk = pd.DataFrame(
        {
            "empty_output": df["output"].str.len() == 0,
            "very_short_output": df["output_words"] <= 1,
            "very_long_output_p95_plus": df["output_words"] >= df["output_words"].quantile(0.95),
            "duplicate_full_text": df["full_text"].duplicated(keep=False),
            "truncated": df.get("truncated", pd.Series(False, index=df.index)).astype(bool),
            "mc_output_not_single_letter": df["multiple_choice"] & ~df["output"].str.match(r"^\s*[A-D](?:[\).:\-])?\s*$", case=False, na=False),
        }
    )
    risk_summary = pd.DataFrame({"count": risk.sum().astype(int), "pct": (risk.mean() * 100).round(2)})
    display(risk_summary.sort_values("pct", ascending=False))

## Beispiele pro Kategorie pruefen

In [ ]:
def show_examples(task: str, n: int = SAMPLE_FOR_DISPLAY, random_state: int = SEED) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    subset = df[df["primary_task"].eq(task)]
    if subset.empty:
        return pd.DataFrame()
    sample = subset.sample(min(n, len(subset)), random_state=random_state)
    cols = ["source", "split", "primary_task", "instruction", "input", "output"]
    out = sample[cols].copy()
    for col in ["instruction", "input", "output"]:
        out[col] = out[col].map(short)
    return out


if not df.empty:
    for task in df["primary_task"].value_counts().head(8).index:
        print(f"\n=== {task} ===")
        display(show_examples(task))

## Zielmix: Ist dein Datensatz breit genug?

In [ ]:
TARGET_MIX = pd.DataFrame(
    [
        {"area": "general_instruction", "min_pct": 25, "max_pct": 45, "why": "breites Befolgen von Anweisungen ohne Benchmark-Overfit"},
        {"area": "qa", "min_pct": 15, "max_pct": 35, "why": "Frageverstaendnis, OpenBookQA-nahe Skills"},
        {"area": "multiple_choice", "min_pct": 10, "max_pct": 25, "why": "Leaderboard parst A/B/C/D; Format muss sitzen"},
        {"area": "bench_hellaswag_like", "min_pct": 8, "max_pct": 18, "why": "Commonsense-Fortsetzungen"},
        {"area": "bench_winogrande_like", "min_pct": 4, "max_pct": 10, "why": "Coreference ist klein, aber WinoGrande ist 25% des Public Avg"},
        {"area": "bench_openbookqa_like", "min_pct": 6, "max_pct": 15, "why": "Science-MC und Faktenreasoning"},
        {"area": "bench_lambada_like", "min_pct": 8, "max_pct": 20, "why": "Long-context Completion nicht durch Chat-SFT zerstoeren"},
        {"area": "chat", "min_pct": 8, "max_pct": 25, "why": "nuetzliches Assistant-Verhalten, aber nicht alles in Chat verwandeln"},
    ]
)

if not df.empty:
    current = {}
    for area in TARGET_MIX["area"]:
        if area in df.columns:
            current[area] = float(df[area].mean() * 100)
        elif area in set(df["primary_task"]):
            current[area] = float((df["primary_task"] == area).mean() * 100)
        else:
            current[area] = 0.0

    target = TARGET_MIX.copy()
    target["current_pct"] = target["area"].map(current).round(2)
    target["target_mid_pct"] = ((target["min_pct"] + target["max_pct"]) / 2).round(2)
    target["delta_to_mid"] = (target["target_mid_pct"] - target["current_pct"]).round(2)
    target["status"] = target.apply(
        lambda r: "low" if r.current_pct < r.min_pct else ("high" if r.current_pct > r.max_pct else "ok"),
        axis=1,
    )
    display(target[["area", "current_pct", "min_pct", "max_pct", "delta_to_mid", "status", "why"]])
else:
    display(TARGET_MIX)

## Best Practices fuer SFT Richtung Benchmarks und breites Modell

Nicht auf die Public-Validation-Beispiele trainieren. Nutze die Public Benchmarks als Diagnose, nicht als Trainingsdaten. Wenn du Daten von denselben Quellen nutzt, decontaminiere gegen die Eval-Splits.

Prioritaeten:

1. Multiple-Choice-Format explizit trainieren. Fuer HellaSwag/OpenBookQA muss die erste Completion ein Buchstabe `A`-`D` sein, fuer WinoGrande `A`/`B`. Baue Beispiele ein, in denen die Antwort wirklich nur `A`, `B`, `C` oder `D` ist. Verbose MC-Antworten koennen im Leaderboard als invalid oder falsch enden.
2. Nicht nur Chat-SFT. OASST1/Dolly machen das Modell hilfreicher, koennen aber Next-Word/LAMBADA und Completion-Stil verschlechtern, wenn sie dominieren. Behalte genug Plain-Text-Completion, kurze Antwort, Cloze und Long-Context-Beispiele.
3. HellaSwag targettet plausiblen naechsten Satz. Gute Daten: Aktivitaet + Kontext + vier Endings, aber auch normale Fortsetzungsdaten aus Narratives/WikiHow/Alltagsablaeufen. Trainiere nicht nur Fakten-QA.
4. WinoGrande braucht Coreference. Ergaenze kleine, saubere binaere Beispiele mit Pronomen, Namen, Objekten, Ursache/Wirkung und expliziten Optionen `A`/`B`.
5. OpenBookQA braucht einfache Science-Fakten plus Reasoning. Gute Beispiele enthalten kurze Frage, vier plausible Optionen, und Antwort im Letter-Format. Erklaerungen gehoeren nicht in die Completion, wenn das Leaderboard nur den Buchstaben parst.
6. LAMBADA braucht Sprachmodell-Faehigkeit. Zu aggressives SFT kann Base-LM-Faehigkeiten ueberschreiben. Tracke weiter `pretrain/val_perplexity` und halte `lambada_like` als OpenWebText Continuation-Guard im Mix.
7. Kleine Modelle profitieren von Konsistenz. Weniger chaotische Templates, weniger extrem lange Outputs, weniger mehrsprachige und code-lastige Daten, wenn diese nicht evaluiert werden.
8. Nutze Source-Balancing statt blindem Concatenation. Ein grosser Chat-Korpus kann kleine, benchmarkrelevante Kategorien verdraengen. Arbeite mit Gewichten oder Caps pro Quelle.
9. Evaluiere ablationsweise. Aendere immer nur eine Sache: MC-Anteil, OASST1-Gewicht, Science-QA-Cap, Cloze-Mix, LR/Epochen. Miss Public Avg plus einzelne Benchmarks, nicht nur SFT-Val-Loss.

Pragmatischer Zielzustand: Das Modell soll zuerst ein stabiles LM bleiben, dann kurze strukturierte Antworten lernen, und erst danach Assistant-Stil. Fuer den Public Avg zaehlt jedes Benchmark ungefaehr gleich; ein SFT-Mix, der nur Chat besser macht, aber LAMBADA/WinoGrande verschlechtert, ist meist kein guter Tradeoff.


## Naechste Schritte

In [ ]:
if not df.empty:
    low_targets = target[target["status"].eq("low")][["area", "current_pct", "min_pct", "delta_to_mid"]]
    high_targets = target[target["status"].eq("high")][["area", "current_pct", "max_pct", "delta_to_mid"]]

    print("Underrepresented areas:")
    display(low_targets)

    print("Overrepresented areas:")
    display(high_targets)

    print("Concrete actions:")
    actions = []
    if "multiple_choice" in set(low_targets["area"]):
        actions.append("Add/cap up MC examples with completions starting with A/B/C/D.")
    if "bench_winogrande_like" in set(low_targets["area"]):
        actions.append("Add binary coreference examples with blanks/pronouns and A/B options.")
    if "bench_lambada_like" in set(low_targets["area"]):
        actions.append("Keep/add lambada_like OpenWebText continuation examples and monitor pretrain perplexity.")
    if "chat" in set(high_targets["area"]):
        actions.append("Reduce chat oversampling or add more non-chat completion/QA data.")
    if not actions:
        actions.append("Mix looks broadly reasonable by heuristic; inspect examples and run benchmark ablations.")
    for a in actions:
        print(f"- {a}")

## Export summaries to Drive

When running on Colab, this writes compact CSV summaries under `DRIVE_ROOT`. It skips cleanly when no dataset was loaded.

In [ ]:
import datetime

if df.empty:
    print("No dataset loaded; nothing to export.")
else:
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    export_dir = DRIVE_ROOT / f"analysis_{stamp}"
    export_dir.mkdir(parents=True, exist_ok=False)

    df.to_csv(export_dir / "classified_rows.csv", index=False)
    task_mix.to_csv(export_dir / "task_mix.csv")
    if "bench_mix" in globals():
        bench_mix.to_csv(export_dir / "benchmark_mix.csv")
    if "target" in globals():
        target.to_csv(export_dir / "target_mix.csv", index=False)

    print(f"Exported analysis summaries to: {export_dir}")